In [1]:
##########################################################################
# Federal University of Pernambuco - PIMES UFPE
# Research: The Impacts of Uncertainty on Economic Activity Across Brazilian States
# Researcher: Paulo Francisco da Silva Junior
# Advisor: Marcelo Eduardo Alves da Silva, PhD.
##########################################################################

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import yfinance as yf

In [3]:
# Define paths: Raw and Processed
pathRaw="..\\Data\\Raw\\Time Series\\"
pathProcessed="..\\Data\\Processed\\Time Series\\"

In [4]:
# Variables to help loop interacion
# We set the short term for each brazilian state
estados = {
    "Bahia":"BA",
    "Ceara":"CE",
    "Espirito Santo":"ES",
    "Goias":"GO",
    "Minas Gerais":"MG",
    "Para":"PA",
    "Parana":"PR",
    "Pernambuco":"PE",
    "Rio de Janeiro":"RJ",
    "Rio Grande do Sul":"RS",
    "Santa Catarina":"SC",
    "Sao Paulo":"SP"
}

In [5]:
# The code bellow does the following steps for each region/state:
# Open data, remove strings and fill with NA
# Filter the range Jan2004 - Dez2019
# Save the adjusted data

In [6]:
for state in estados:
    dados = pd.read_csv(str(pathRaw)+str(state)+"_IEF_IBC.csv", sep=";", encoding="latin-1", decimal=",", skipfooter=1)
    dados.columns=["Data","IBCR-"+estados[state]+"-Desazonalizado","IEF-"+estados[state]]
    dados["Data"] = pd.to_datetime(dados["Data"])
    dados.replace("-",np.nan, inplace=True)
    dados.dropna(inplace=True)
    dados.reset_index(inplace=True,drop=True)
    dados = dados[(dados["Data"] >= "2004-01-01") & (dados["Data"]<= "2019-12-31")]
    for coluna in dados.columns:
        if(coluna=="Data"):
            next
        else:
            #dados[coluna] = dados[coluna].str.replace(" ","")
            #dados[coluna] = dados[coluna].str.replace(",",".")
            dados[coluna] = dados[coluna].astype(float)
    dados.to_excel(str(pathProcessed)+str(state)+"_IEF_IBC"+".xlsx")

C:\Users\Paulo\AppData\Local\Temp\ipykernel_9900\3889624636.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  dados = pd.read_csv(str(pathRaw)+str(state)+"_IEF_IBC.csv", sep=";", encoding="latin-1", decimal=",", skipfooter=1)
C:\Users\Paulo\AppData\Local\Temp\ipykernel_9900\3889624636.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dados["Data"] = pd.to_datetime(dados["Data"])
C:\Users\Paulo\AppData\Local\Temp\ipykernel_9900\3889624636.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  dados = pd.read_csv(str(pathRaw)+str(state)+"_IEF_IBC.csv", sep=";", encoding="latin-1", decimal=",", skipfooter=1)
C:\Users\Pau

In [7]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Data                    192 non-null    datetime64[us]
 1   IBCR-SP-Desazonalizado  192 non-null    float64       
 2   IEF-SP                  192 non-null    float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 4.6 KB


In [8]:
# Treat the Industrial Production Data

In [9]:
# According with the following steps: 
# Read data, transpose data, reset index, drop index, rename columns, adjust date range
dados = pd.read_excel(str(pathRaw)+"PIM_PF_Desazonalizado.xlsx", skiprows=4, skipfooter=1)
dados = dados.transpose()
dados.reset_index(inplace=True)
dados.columns=dados[:1].values[0]
dados.drop(0,inplace=True)
dados.rename(columns={"Unnamed: 0": "Data", "Pará": "Para", "Ceará":"Ceara", "Espírito Santo": "Espirito Santo", "São Paulo": "Sao Paulo", "Paraná":"Parana", "Goiás":"Goias"}, inplace=True)
dados["Data"] = pd.date_range("2002-01-01", "2025-01-01", freq="ME")
dados = dados[(dados["Data"]>= "2004-01-01") & (dados["Data"] <="2019-12-31")]
dados.reset_index(inplace=True,drop=True)

In [10]:
# Create a temporary list to merge all the states name
temp = ["Data"]
for x in list(estados.keys()):
    temp.append(x)

dados = dados[temp]

In [11]:
# Transform to float data type
for coluna in dados.columns:
        if(coluna=="Data"):
            next
        else:
            dados[coluna] = dados[coluna].astype(float)
dados.to_excel(str(pathProcessed)+"PIM_PF_Desazonalizado.xlsx")

In [12]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Data               192 non-null    datetime64[us]
 1   Bahia              192 non-null    float64       
 2   Ceara              192 non-null    float64       
 3   Espirito Santo     192 non-null    float64       
 4   Goias              192 non-null    float64       
 5   Minas Gerais       192 non-null    float64       
 6   Para               192 non-null    float64       
 7   Parana             192 non-null    float64       
 8   Pernambuco         192 non-null    float64       
 9   Rio de Janeiro     192 non-null    float64       
 10  Rio Grande do Sul  192 non-null    float64       
 11  Santa Catarina     192 non-null    float64       
 12  Sao Paulo          192 non-null    float64       
dtypes: datetime64[us](1), float64(12)
memory usage: 19.6 KB


In [13]:
# Handle with interest rate

In [14]:
# Accoring with the following steps:
# Read data, transform to the right datetime and data type, group by month trought the mean metric for the adjusted period
dados = pd.read_csv(str(pathRaw)+"Taxa de juros - Meta Selic.csv", encoding="utf-8", sep=";", skipfooter=1, decimal=",")
dados["Data"] = pd.to_datetime(dados["Data"],format="%d/%m/%Y")
dados.columns=["Data","Selic_Meta"]
dados["Selic_Meta"] = dados["Selic_Meta"].astype(float)
dados["Data"] = dados["Data"].dt.strftime("%Y/%m")
dados = pd.DataFrame(dados.groupby("Data").mean().reset_index())
dados = dados[(dados["Data"]>= "2004-01-01") & (dados["Data"] <="2020-01-31")]
dados.to_excel(str(pathProcessed)+"Selic_Meta.xlsx")

C:\Users\Paulo\AppData\Local\Temp\ipykernel_9900\2248915729.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  dados = pd.read_csv(str(pathRaw)+"Taxa de juros - Meta Selic.csv", encoding="utf-8", sep=";", skipfooter=1, decimal=",")


In [15]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Data        192 non-null    str    
 1   Selic_Meta  192 non-null    float64
dtypes: float64(1), str(1)
memory usage: 3.1 KB


In [16]:
# Handle ibovespa index data

In [17]:
# Accoring with the following steps:
# Read data, group by month trought the mean metric for the adjusted period
# OBS: Depeding on your region you may need to use a VPN or proxy
dados= yf.Ticker("^BVSP")
dados = pd.DataFrame(yf.download("^BVSP", start="2004-01-01", end="2020-01-01")["Close"])
dados.reset_index(inplace=True)
dados["Data"] = dados["Date"].dt.strftime("%Y/%m")
dados = pd.DataFrame(dados.groupby("Data").mean().reset_index())
dados.drop(columns=["Date"],inplace=True)
dados.columns=["Data","BVSP"]
dados["Data"] = pd.to_datetime(dados["Data"])
dados.to_excel(str(pathProcessed)+"BVSP.xlsx")

[*********************100%***********************]  1 of 1 completed
C:\Users\Paulo\AppData\Local\Temp\ipykernel_9900\4267126674.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dados["Data"] = pd.to_datetime(dados["Data"])


In [18]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Data    192 non-null    datetime64[us]
 1   BVSP    192 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 3.1 KB


In [19]:
# Handle the IIE-Br data

In [20]:
# Accoring with the following steps:
# Read data, transform to the right datetime, adjust the period and transform to the right data type
dados = pd.read_csv(str(pathRaw)+"IIE-Br.csv", sep=";", decimal=",")
dados.columns = ["Data", "IIE-Br"]
dados["Data"] = pd.date_range("2000-01-01","2025-02-01",freq="ME")
dados = dados[(dados["Data"]>="2004-01") & (dados["Data"]<="2020-01")]
dados.reset_index(drop=True)
dados["IIE-Br"] = dados["IIE-Br"].astype(float)
dados.to_excel(str(pathProcessed)+"IIE-Br.xlsx")

In [21]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 48 to 239
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Data    192 non-null    datetime64[us]
 1   IIE-Br  192 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 3.1 KB


In [22]:
# Handle EPU Br data

In [23]:
# Accoring with the following steps:
# Read data, transform to the right datetime, adjust the period and transform to the right data type
dados = pd.read_excel(str(pathRaw)+"Brazil_Policy_Uncertainty_Data.xlsx", skipfooter=1)
dados["Data"] = pd.date_range("1991-01", "2025-02",freq="ME")
dados = dados[(dados["Data"]>="2004-01") & (dados["Data"]<="2020-01") ]
dados.drop(columns=["year","month"], inplace=True)
dados.columns=["EPU_Br","Data"]
dados.reset_index(drop=True, inplace=True)
dados.to_excel(str(pathProcessed)+"EPU_Br.xlsx")

In [24]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   EPU_Br  192 non-null    float64       
 1   Data    192 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1)
memory usage: 3.1 KB


In [25]:
# Handle GEPU (Global EPU)

In [26]:
# Accoring with the following steps:
# Read data, transform to the right datetime, adjust the period and transform to the right data type
dados = pd.read_excel(str(pathRaw)+"Global_Policy_Uncertainty_Data.xlsx", skipfooter=2)
dados["Data"] = pd.date_range("1997-01", "2025-02",freq="ME")
dados = dados[(dados["Data"]>="2004-01") & (dados["Data"]<="2020-01") ]
dados.drop(columns=["Year","Month","GEPU_current"], inplace=True)
dados.columns=["GEPU_PPP","Data"]
dados.reset_index(drop=True, inplace=True)
dados.to_excel(str(pathProcessed)+"GEPU_PPP.xlsx")

In [27]:
# Check out the result
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   GEPU_PPP  192 non-null    float64       
 1   Data      192 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1)
memory usage: 3.1 KB
